In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.data.market_loader import MarketLoader

from src.curves.curve_snapshot import CurveSnapshot
from src.curves.bootstrap.bootstrap_engine import BootstrapCurveEngine
from src.curves.projection_curve import ProjectionCurve

from src.instruments.instrument_builder import InstrumentBuilder

from src.schedules.payment_schedule import PaymentSchedule

from src.trades.interest_rate_swap import InterestRateSwap

from src.pricing.swap_pricer import SwapPricer

In [2]:
# payment scheduler
payment_schedule = PaymentSchedule(
    maturity = 5.0,
    frequency = 'quarterly'
)

payment_schedule.generate_schedule()

[0.25,
 0.5,
 0.75,
 1.0,
 1.25,
 1.5,
 1.75,
 2.0,
 2.25,
 2.5,
 2.75,
 3.0,
 3.25,
 3.5,
 3.75,
 4.0,
 4.25,
 4.5,
 4.75,
 5.0]

In [3]:
# IRS trade object
IR_swap = InterestRateSwap(
    notional = 1_000_000,
    maturity = 3.0,
    fixed_rate = 3.70,
    pay_fixed = True    # pay-fixed/receive-floating vanilla IRS
)

display(IR_swap)

IR_swap.summary()

InterestRateSwap(Notional=1000000,  Maturity=3.0,  FixedRate=3.7,  Direction=PAY_FIXED)

{'Notional': 1000000,
 'Maturity': 3.0,
 'FixedRate': 3.7,
 'Direction': 'PAY_FIXED',
 'Currency': 'USD',
 'FixedFrequency': 'semiannual',
 'FloatFrequency': 'quarterly'}

In [4]:
# downloading market curves
loader = MarketLoader()
market_curves = loader.market_loader_pipeline()


treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..
estr curve dataset already downloaded..


In [5]:
# downloading swap curves
swap_curves = loader.swap_loader_pipeline()

usd_ois curve dataset already downloaded..
eur_ois curve dataset already downloaded..


In [6]:
### create curve snapshots
# SOFR snapshot
sofr_df = market_curves['sofr']

latest_date = sofr_df.index[-1]
latest_sofr_curve = sofr_df.iloc[-1]

sofr_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'sofr',
    as_of_date = latest_date,
    curve_row = latest_sofr_curve
)

# Futures snapshot
future_df = market_curves['futures']
latest_futures_curve = future_df.iloc[-1]

futures_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'futures',
    as_of_date = latest_date,
    curve_row = latest_futures_curve
)

# OIS snapshot
ois_df = swap_curves['usd_ois']

latest_swap_date = ois_df.index[-1]
latest_swap_curve = ois_df.iloc[-1]

ois_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'usd_ois',
    as_of_date = latest_swap_date,
    curve_row = latest_swap_curve
)

In [7]:
### create instruments from curve snapshot
# deposits
deposit_instruments = InstrumentBuilder.build_deposit_instruments(snapshot = sofr_snapshot)

# futures
future_instruments = InstrumentBuilder.build_future_instruments(snapshot = futures_snapshot)

# ois
ois_instruments = InstrumentBuilder.build_ois_instruments(snapshot = ois_snapshot)

In [8]:
# bootstrapping engine for generating the discount curve
all_instruments = deposit_instruments + future_instruments + ois_instruments

engine = BootstrapCurveEngine()

discount_curve = engine.bootstrap(
    snapshot = sofr_snapshot,
    instruments = all_instruments
)

# projection curve
projection_curve = ProjectionCurve(discount_curve = discount_curve)

In [9]:
# vanilla IRS pricer
pricer = SwapPricer(
    discount_curve = discount_curve,
    projection_curve = projection_curve
)

pricer.valuation_report(swap = IR_swap)

,Metric,Value
0,Fixed-leg PV,104123.616560
1,Floating-leg PV,103539.575439
2,Swap PV,-584.041121


In [ ]:
# swap par rate
par_rate = pricer.par_rate(swap = IR_swap)

print(f'Trade fixed rate: {IR_swap.fixed_rate}')
print(f'Par swap rate: {(par_rate * 100):.4f}')

expected_pv = 'positive' if IR_swap.fixed_rate < par_rate * 100 else 'negative'

print(f'The trade should have {expected_pv} PV')

# fixed_leg annuity
schedule = IR_swap.fixed_schedule.generate_schedule()

accrual = 1.0 / PaymentSchedule.FREQUENCY_MAP[IR_swap.fixed_freq]

annuity = 0.0

for t in schedule:
    annuity += accrual * discount_curve.get_discount_factor(maturity = t)

print(f'Swap annuity for a {int(IR_swap.maturity)}Y {IR_swap.direction.lower()} IR swap : {annuity:.2f}')

Trade fixed rate: 3.7
Par swap rate: 3.6793
The trade should have negative PV
Swap annuity for a 3Y pay_fixed IR swap : 2.81
